### Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm


In [2]:
df = pd.read_csv("medicaid-provider-spending.csv")
print(df.columns.tolist())  # column names

C:\Users\David\AppData\Local\Temp\ipykernel_15552\2520810276.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("medicaid-provider-spending.csv")


['BILLING_PROVIDER_NPI_NUM', 'SERVICING_PROVIDER_NPI_NUM', 'HCPCS_CODE', 'CLAIM_FROM_MONTH', 'TOTAL_UNIQUE_BENEFICIARIES', 'TOTAL_CLAIMS', 'TOTAL_PAID']


### filter on codes provided by Clinicans

In [3]:
codes_df = pd.read_excel("Codes Manual.xlsx")

codes = (
    codes_df["CODE"]
    .astype(str)
    .str.strip()
    .dropna()
    .unique()
)

df["HCPCS_CODE"] = df["HCPCS_CODE"].astype(str).str.strip()

filtered_df = df[df["HCPCS_CODE"].isin(codes)].copy()

In [4]:
filtered_df["CLAIM_FROM_MONTH"] = pd.to_datetime(filtered_df["CLAIM_FROM_MONTH"])

filtered_df.sample(20)

,BILLING_PROVIDER_NPI_NUM,SERVICING_PROVIDER_NPI_NUM,HCPCS_CODE,CLAIM_FROM_MONTH,TOTAL_UNIQUE_BENEFICIARIES,TOTAL_CLAIMS,TOTAL_PAID
12567614,1891732210,1891732210,A0427,2018-03-01,71,80,10348.41
160043378,1912567884,1073298634,96127,2024-08-01,64,64,146.97
126426366,1639172851,1124246954,71260,2023-12-01,14,15,451.94
9824796,1972809051,1972809051,A0428,2022-08-01,476,1719,13630.00
39895690,1245623792,1245260470,95004,2021-11-01,15,15,2963.69
169916198,1013523430,1790190189,93010,2024-03-01,13,13,90.09
207135623,1437377454,1962281527,80061,2024-10-01,44,44,0.00
160846956,1184793929,1184793929,81025,2018-11-01,21,21,141.96
138564830,1598071243,1942269311,93010,2024-09-01,41,46,321.09
201941815,1629456900,NaN,86140,2023-07-01,21,21,0.00


### Check join status

In [5]:
codes_set = set(codes_df["CODE"].astype(str).str.strip())
claims_set = set(filtered_df["HCPCS_CODE"].astype(str).str.strip())

missing_codes = codes_set - claims_set

print("total codes: {}".format(len(codes_set)))
print("missing codes: {}".format(len(missing_codes)))


total codes: 303
missing codes: 67


In [6]:
missing_codes_df = (
    pd.DataFrame({"HCPCS_CODE": list(missing_codes)})
    .assign(HCPCS_CODE=lambda d: d["HCPCS_CODE"].astype(str).str.strip())
    .merge(
        codes_df.assign(CODE=codes_df["CODE"].astype(str).str.strip()),
        left_on="HCPCS_CODE",
        right_on="CODE",
        how="left"
    )
    .drop(columns=["CODE"])
)

missing_codes_df

,HCPCS_CODE,Description,Category,Unnamed: 3,Unnamed: 4
0,85258,Clotting factors (selective use),FIBRINOLYSIS/CLOT STABILITY,NaN,NaN
1,84073,anaerobic culture using material from any sour...,NEGATIVE CONTROL,NaN,NaN
2,85252,Clotting factors (selective use),FIBRINOLYSIS/CLOT STABILITY,NaN,NaN
3,71253,CT Chest,CARDIAC IMAGING,NaN,NaN
4,71258,CT Chest,CARDIAC IMAGING,NaN,NaN
...,...,...,...,...,...
62,85242,Clotting factors (selective use),FIBRINOLYSIS/CLOT STABILITY,NaN,NaN
63,71259,CT Chest,CARDIAC IMAGING,NaN,NaN
64,82586,Leukotriene E4,URINARY MEDIATORS,NaN,NaN
65,38530,"Biopsy or excision of lymph node(s); open, in...",MISC,NaN,NaN


### Join

In [7]:
filtered_df["HCPCS_CODE"] = filtered_df["HCPCS_CODE"].astype(str).str.strip()
codes_df["CODE"] = codes_df["CODE"].astype(str).str.strip()

retsef_meta = codes_df[["CODE", "Description"]].drop_duplicates()

filtered_with_meta = filtered_df.merge(
    retsef_meta,
    left_on="HCPCS_CODE",
    right_on="CODE",
    how="left"
).drop(columns=["CODE"])

monthly = (
    filtered_with_meta
    .groupby(["HCPCS_CODE", "CLAIM_FROM_MONTH"], as_index=False)
    .agg({
        "Description": "first",
        "TOTAL_PAID": "sum",
        "TOTAL_CLAIMS": "sum",
        "TOTAL_UNIQUE_BENEFICIARIES": "sum"
    })
    .sort_values(["HCPCS_CODE", "CLAIM_FROM_MONTH"])
)

monthly

,HCPCS_CODE,CLAIM_FROM_MONTH,Description,TOTAL_PAID,TOTAL_CLAIMS,TOTAL_UNIQUE_BENEFICIARIES
0,0295T,2018-03-01,Extended cardiac monitoring,50.58,14,14
1,0295T,2018-04-01,Extended cardiac monitoring,388.26,15,14
2,0295T,2018-05-01,Extended cardiac monitoring,0.00,14,14
3,0295T,2018-06-01,Extended cardiac monitoring,610.95,36,26
4,0295T,2018-08-01,Extended cardiac monitoring,0.00,33,32
...,...,...,...,...,...,...
15794,A0434,2024-08-01,Specialty Care Transport (SCT),6267198.31,10187,6922
15795,A0434,2024-09-01,Specialty Care Transport (SCT),6491171.16,14550,7357
15796,A0434,2024-10-01,Specialty Care Transport (SCT),6835805.15,9502,6076
15797,A0434,2024-11-01,Specialty Care Transport (SCT),5687362.10,7074,4264


### Pivot

In [8]:
claims_pivot = (
    monthly
    .pivot(index="CLAIM_FROM_MONTH", columns="HCPCS_CODE", values="TOTAL_CLAIMS")
    .sort_index()
)

claims_pivot

HCPCS_CODE,0295T,0296T,0297T,0298T,0464T,11100,11101,11102,11103,11104,...,A0422,A0425,A0426,A0427,A0428,A0429,A0430,A0431,A0433,A0434
CLAIM_FROM_MONTH,,,,,,,,,,,,,,,,,,,,,
2018-01-01,NaN,45.0,566.0,88.0,NaN,10190.0,1047.0,NaN,NaN,NaN,...,26536.0,1337484.0,56196.0,399562.0,294462.0,336967.0,1622.0,4857.0,2357.0,41392.0
2018-02-01,NaN,108.0,705.0,192.0,NaN,9802.0,1176.0,NaN,NaN,NaN,...,24168.0,1221412.0,58770.0,347596.0,255220.0,295203.0,1373.0,3621.0,1790.0,37541.0
2018-03-01,14.0,112.0,908.0,276.0,NaN,11276.0,1254.0,NaN,NaN,NaN,...,25256.0,1291656.0,64234.0,367339.0,279179.0,305396.0,1570.0,4776.0,2076.0,39262.0
2018-04-01,15.0,106.0,889.0,191.0,NaN,11231.0,1243.0,NaN,NaN,NaN,...,23174.0,1277498.0,74654.0,360310.0,271567.0,296735.0,1295.0,5234.0,1874.0,45630.0
2018-05-01,14.0,188.0,1048.0,183.0,NaN,12689.0,1431.0,NaN,NaN,NaN,...,24006.0,1348400.0,77428.0,380685.0,284264.0,315870.0,1375.0,5575.0,1768.0,42598.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11542.0,1197.0,1034.0,...,15116.0,1708899.0,12554.0,371576.0,226288.0,378416.0,1395.0,5773.0,2858.0,10187.0
2024-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9255.0,831.0,853.0,...,14242.0,1592956.0,12063.0,341467.0,201599.0,348772.0,1113.0,4879.0,2394.0,14550.0
2024-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9979.0,967.0,1070.0,...,13156.0,1549316.0,11433.0,309061.0,201509.0,329905.0,1018.0,4737.0,2155.0,9502.0


### Cutoff past end of 2023

In [9]:
full_months = pd.date_range(
    monthly["CLAIM_FROM_MONTH"].min(),
    monthly["CLAIM_FROM_MONTH"].max(),
    freq="MS"
)
claims_pivot = claims_pivot.reindex(full_months)
claims_pivot.index.name = "CLAIM_FROM_MONTH"
claims_pivot

HCPCS_CODE,0295T,0296T,0297T,0298T,0464T,11100,11101,11102,11103,11104,...,A0422,A0425,A0426,A0427,A0428,A0429,A0430,A0431,A0433,A0434
CLAIM_FROM_MONTH,,,,,,,,,,,,,,,,,,,,,
2018-01-01,NaN,45.0,566.0,88.0,NaN,10190.0,1047.0,NaN,NaN,NaN,...,26536.0,1337484.0,56196.0,399562.0,294462.0,336967.0,1622.0,4857.0,2357.0,41392.0
2018-02-01,NaN,108.0,705.0,192.0,NaN,9802.0,1176.0,NaN,NaN,NaN,...,24168.0,1221412.0,58770.0,347596.0,255220.0,295203.0,1373.0,3621.0,1790.0,37541.0
2018-03-01,14.0,112.0,908.0,276.0,NaN,11276.0,1254.0,NaN,NaN,NaN,...,25256.0,1291656.0,64234.0,367339.0,279179.0,305396.0,1570.0,4776.0,2076.0,39262.0
2018-04-01,15.0,106.0,889.0,191.0,NaN,11231.0,1243.0,NaN,NaN,NaN,...,23174.0,1277498.0,74654.0,360310.0,271567.0,296735.0,1295.0,5234.0,1874.0,45630.0
2018-05-01,14.0,188.0,1048.0,183.0,NaN,12689.0,1431.0,NaN,NaN,NaN,...,24006.0,1348400.0,77428.0,380685.0,284264.0,315870.0,1375.0,5575.0,1768.0,42598.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11542.0,1197.0,1034.0,...,15116.0,1708899.0,12554.0,371576.0,226288.0,378416.0,1395.0,5773.0,2858.0,10187.0
2024-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9255.0,831.0,853.0,...,14242.0,1592956.0,12063.0,341467.0,201599.0,348772.0,1113.0,4879.0,2394.0,14550.0
2024-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9979.0,967.0,1070.0,...,13156.0,1549316.0,11433.0,309061.0,201509.0,329905.0,1018.0,4737.0,2155.0,9502.0


In [10]:
cutoff = full_months.max() - pd.DateOffset(months=12)

claims_pivot = claims_pivot.loc[claims_pivot.index <= cutoff]

claims_pivot

HCPCS_CODE,0295T,0296T,0297T,0298T,0464T,11100,11101,11102,11103,11104,...,A0422,A0425,A0426,A0427,A0428,A0429,A0430,A0431,A0433,A0434
CLAIM_FROM_MONTH,,,,,,,,,,,,,,,,,,,,,
2018-01-01,NaN,45.0,566.0,88.0,NaN,10190.0,1047.0,NaN,NaN,NaN,...,26536.0,1337484.0,56196.0,399562.0,294462.0,336967.0,1622.0,4857.0,2357.0,41392.0
2018-02-01,NaN,108.0,705.0,192.0,NaN,9802.0,1176.0,NaN,NaN,NaN,...,24168.0,1221412.0,58770.0,347596.0,255220.0,295203.0,1373.0,3621.0,1790.0,37541.0
2018-03-01,14.0,112.0,908.0,276.0,NaN,11276.0,1254.0,NaN,NaN,NaN,...,25256.0,1291656.0,64234.0,367339.0,279179.0,305396.0,1570.0,4776.0,2076.0,39262.0
2018-04-01,15.0,106.0,889.0,191.0,NaN,11231.0,1243.0,NaN,NaN,NaN,...,23174.0,1277498.0,74654.0,360310.0,271567.0,296735.0,1295.0,5234.0,1874.0,45630.0
2018-05-01,14.0,188.0,1048.0,183.0,NaN,12689.0,1431.0,NaN,NaN,NaN,...,24006.0,1348400.0,77428.0,380685.0,284264.0,315870.0,1375.0,5575.0,1768.0,42598.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15312.0,1461.0,1334.0,...,21289.0,1721805.0,14977.0,459143.0,260018.0,451244.0,1946.0,8221.0,3675.0,11273.0
2023-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11959.0,1274.0,1099.0,...,20759.0,1624976.0,14177.0,432132.0,241175.0,430239.0,1781.0,8385.0,3318.0,24207.0
2023-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13886.0,1294.0,1225.0,...,20536.0,1690205.0,14425.0,426485.0,246089.0,441044.0,1753.0,8155.0,3620.0,33807.0


In [11]:
claims_pivot.to_csv("claims_pivot.csv")

In [13]:
missing_codes_df.to_csv("missing_codes.csv", index=False)